<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_72_computer_use_browser_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🖱️ **Module 72 — Computer-use / Browser Agents** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 9 · Production Gaps


# Module 72 — Computer-use / Browser Agents

> Every app has an API. **Except when it doesn't.** Internal admin panels, legacy SaaS, banking portals, government forms, that one Salesforce screen — they're all human-only UIs. **Computer-use agents** close the gap: a vision-language model (M65) watches a screen, reasons about what to click, and emits `click(x, y)` / `type("...")` / `scroll` actions. **Browser agents** are the same thing scoped to a web page (cheaper, sandboxable, faster).
>
> This module is the map of the space: **Claude computer-use**, **OpenAI Operator / Computer-use Agent**, **Playwright-AI / Browser Use / browser-use** for OSS, and the safety patterns that keep them from wiring your savings to a stranger.

### What you'll cover
1. Why computer-use exists — when APIs aren't an option
2. The mental model: **screenshot → VLM → action → loop**
3. The standard **action space**
4. The 2025-26 hosted providers: **Claude · OpenAI Operator · Gemini**
5. **Browser-use / Skyvern / WebVoyager** — open-source browser agents
6. **Playwright + LLM** — the build-it-yourself recipe
7. Benchmarks: **OSWorld · WebArena · Mind2Web · ScreenSpot · WebVoyager-Eval**
8. **Sandboxing + safety** — VM isolation, allowlists, human-in-the-loop
9. Failure modes (and the ones that haunt you most)
10. Decision tree — pick an agent stack


## 1 · Why computer-use exists

| Situation | Why not an API? |
|---|---|
| Internal admin portal | nobody wrote the API; no time / no budget |
| Legacy SaaS | API is paid tier; you're on the free tier |
| Government / bank portal | "we have an API" → 50-page PDF + manual onboarding |
| Salesforce or Workday screen | API exists but auth is locked down by IT |
| Multi-system workflow | each system has an API; the workflow doesn't |
| Verifying a UI change | the *human-visible* output is what matters |

The agent **sees what the human sees** and **acts like the human acts** — every system that exists for humans becomes scriptable, with no integration work. That's the value prop. The cost prop is **fragility + safety risk**, both of which we'll spend the rest of the module on.

## 2 · The mental model

```
   ┌──────────────────────────────────────────────────────────┐
   │  loop:                                                    │
   │    screenshot     ←─ OS / Playwright / Chrome DevTools    │
   │      │                                                    │
   │      ▼                                                    │
   │    VLM(screenshot, task, history)                         │
   │      │  reasons about pixels + accumulated context        │
   │      ▼                                                    │
   │    action  ──► click(x,y) / type("…") / scroll / key / done│
   │      │                                                    │
   │      ▼                                                    │
   │    execute action on the OS / browser                     │
   │      │                                                    │
   │      ▼                                                    │
   │    sleep N ms (let the UI settle); back to top            │
   └──────────────────────────────────────────────────────────┘
```

Each turn is **one screenshot → one action**. The VLM has to:
1. **Locate** the target element from pixels (icon, button, input field).
2. **Predict precise coordinates** (or pick from a numbered overlay).
3. **Plan** the next action consistent with the task.
4. **Detect completion** ("the form is submitted; emit `done`").

The whole system is a tight feedback loop running at ~1–3 actions / second. Long tasks = hundreds of iterations.

## 3 · The action space

Every computer-use API ships roughly the same primitives. Below is the Claude / OpenAI shape:

| Tool | Args | Effect |
|---|---|---|
| `screenshot` | — | return the current screen (PNG bytes) |
| `mouse_move` | `(x, y)` | cursor only |
| `left_click` | `(x, y)` | click at coords |
| `right_click` / `middle_click` / `double_click` | `(x, y)` | variant clicks |
| `left_click_drag` | `(x0, y0) → (x1, y1)` | drag selection |
| `type` | `"text"` | type into the focused element |
| `key` | `"return"` / `"ctrl+a"` / `"cmd+v"` | named key combos |
| `scroll` | `(dx, dy)` or named direction | scroll |
| `wait` | `ms` | sleep |
| `cursor_position` | — | current mouse position |

Browser-scoped agents add: `navigate(url)`, `back`, `forward`, `tab_open` / `tab_close`. They **drop** OS-level primitives (no `cmd+space` to launch an app).

## 4 · The 2025-26 hosted providers

In [ ]:
# Hosted computer-use APIs (latest as of 2025-26)

# 1) ANTHROPIC — Claude computer-use (built-in tool type)
anthropic_sketch = '''
import anthropic, base64
client = anthropic.Anthropic()

def take_screenshot() -> bytes:
    # pyautogui / Playwright / VM-side capture
    ...

screenshot_b64 = base64.b64encode(take_screenshot()).decode()

resp = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    tools=[
        {"type": "computer_20241022", "name": "computer",
         "display_width_px": 1280, "display_height_px": 800, "display_number": 1},
    ],
    messages=[
        {"role": "user", "content": [
            {"type": "text", "text": "Open Gmail and find the unread message from Alice."},
            {"type": "image",
             "source": {"type": "base64", "media_type": "image/png", "data": screenshot_b64}},
        ]},
    ],
)
# Then loop: parse tool_use blocks (action), execute, screenshot again, send back, repeat.
'''
print(anthropic_sketch)

In [ ]:
# 2) OPENAI — Computer-use Agent (Operator) via the Responses API
openai_sketch = '''
from openai import OpenAI
client = OpenAI()

resp = client.responses.create(
    model="computer-use-preview",
    tools=[{"type": "computer-use_preview",
            "display_width": 1280, "display_height": 800, "environment": "browser"}],
    input=[{
        "role": "user",
        "content": [{"type": "input_text", "text": "Book the cheapest flight CDG → LHR for next Friday."}],
    }],
    truncation="auto",
)
# The model returns "computer_call" actions in the response object.
# You execute them and call back with the new screenshot.
'''
print(openai_sketch)

**Two more options worth knowing.**
- **Gemini 2.5 Computer Use / Mariner** (Google) — same idea, in Vertex AI.
- **Anthropic Claude Code + computer-use** — the agentic IDE pattern; the model can drive your browser to verify a code change.

All three providers ship **reference loops** in their cookbooks. Don't write the screenshot/action loop yourself — copy theirs and customise.

## 5 · Open-source browser agents

When you don't want to send screenshots to a vendor, or you want **deterministic browser primitives** instead of OS-level pixel control, the OSS stack:

| Project | What it is | Notes |
|---|---|---|
| **browser-use** | Python wrapper: LLM drives a Playwright browser via DOM + accessibility tree | most popular OSS in 2025; pluggable models (OpenAI/Anthropic/local) |
| **Skyvern** | end-to-end browser-agent platform; visual + DOM hybrid | self-hostable; commercial cloud too |
| **WebVoyager** | research agent that broke many WebArena scores | GPT-4V + retrieval over DOM |
| **AgentE / Multi-On / Auto-GPT-browser** | older waves; mostly superseded | |
| **Stagehand (Browserbase)** | "natural-language Playwright" — `page.act("Click sign-in")` | clean DX; hosted browser sandbox |
| **Magnitude / Roboflow Inference** | screenshot-only OSS computer-use | |
| **AutoUI / SeeClick / OS-Atlas / UI-TARS** | small open VLMs trained for UI grounding | self-host the brain too |

**The 2025 default OSS recipe:** **`browser-use`** + **GPT-4o** or **Claude 3.5 Sonnet** for the brain, **Playwright** under the hood, in a Docker container with a fresh user profile per task.

In [ ]:
# browser-use minimal example
browser_use_sketch = '''
# pip install browser-use playwright && playwright install chromium
import asyncio
from browser_use import Agent, ChatOpenAI

agent = Agent(
    task="Find the price of the cheapest 256 GB iPhone 17 on the Apple India website.",
    llm=ChatOpenAI(model="gpt-4o"),
)

async def main():
    result = await agent.run()
    print(result.final_result())

asyncio.run(main())
'''
print(browser_use_sketch)

## 6 · Playwright + LLM — build it yourself

In [ ]:
# Pseudocode for the cleanest self-built browser agent
playwright_loop = '''
from playwright.async_api import async_playwright
import base64, asyncio
from openai import AsyncOpenAI                    # or anthropic, or any vLLM endpoint

client = AsyncOpenAI()

SYSTEM = (
    "You are a browser-use agent. At each step you are given a screenshot "
    "and a task. Reply ONLY with one of:\n"
    "- CLICK x y\n- TYPE \"text\"\n- SCROLL dy\n- KEY name\n"
    "- NAVIGATE url\n- DONE  (when the task is achieved)\n"
    "Output nothing else."
)

async def step(page, task, history):
    png = await page.screenshot()
    b64 = base64.b64encode(png).decode()
    msg = await client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM},
            *history,
            {"role": "user", "content": [
                {"type": "text", "text": f"Task: {task}\nReply with one action."},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ]},
        ],
    )
    return msg.choices[0].message.content.strip()

async def run(task: str, start_url: str, max_steps: int = 25):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto(start_url)

        history = []
        for i in range(max_steps):
            action = await step(page, task, history)
            print(i, action)
            if action.startswith("DONE"): break
            history.append({"role": "assistant", "content": action})

            tok, *rest = action.split()
            if tok == "CLICK":
                x, y = map(int, rest);  await page.mouse.click(x, y)
            elif tok == "TYPE":
                await page.keyboard.type(action.split('"',1)[1].rstrip('"'))
            elif tok == "SCROLL":
                await page.mouse.wheel(0, int(rest[0]))
            elif tok == "KEY":
                await page.keyboard.press(rest[0])
            elif tok == "NAVIGATE":
                await page.goto(rest[0])
            await page.wait_for_timeout(700)

        await browser.close()

# asyncio.run(run("buy the cheapest copy of Crime and Punishment",
#                 "https://www.amazon.com"))
'''
print(playwright_loop)

**Why building it yourself is sometimes the right call.**
- You control the **action vocabulary** — add `extract_text`, `read_aria_tree`, `wait_for_selector`.
- You control the **stop condition** — your own scorer, your own retries.
- You can **swap the model** — local Qwen2.5-VL when cheap, GPT-4o when cost is no object.
- You see every screenshot + action — easier to **debug** than a hosted black box.

**Why building it yourself is sometimes the wrong call.**
- Hosted providers train the model specifically for accurate UI grounding. Their accuracy is hard to match.
- They give you ready-made loops, retry/timeout policies, and screenshot trimming.

## 7 · Benchmarks

How good *is* the state of the art? As of 2025-26:

| Benchmark | What it measures | Frontier headline number |
|---|---|---|
| **OSWorld** | full Linux desktop tasks (file manager, email, spreadsheets, browsers) | ~30-45% success — still far from human |
| **WebArena** | full-fidelity web apps (GitLab, Shopping, Reddit) | ~40-55% |
| **WebVoyager** | open-web tasks across 15 real sites | ~70-85% |
| **Mind2Web** | one-shot web tasks from real sites | element-accuracy 75%+ |
| **ScreenSpot / ScreenSpot-v2** | "click the *Submit* button" in a screenshot — pure UI grounding | 90%+ for frontier VLMs |
| **AndroidWorld** | Android UI tasks | <40% |
| **VisualWebArena** | requires reading charts / images on the page | <40% |

**Two lessons.**
1. **Pure grounding (ScreenSpot)** is solved. Models can find elements at high accuracy.
2. **Full tasks** (OSWorld, WebArena) are not — long horizons, sub-goal planning, and recovery from misclicks dominate the failure modes.

Run agents on your own internal tasks before believing leaderboard numbers — task distributions differ wildly from your real workload.

## 8 · Sandboxing + safety — the only section that really matters

If you take nothing else from this module, take this: **a computer-use agent will eventually click something you didn't intend.** Plan for it.

### Three layers of defence

**Layer 1 — VM isolation.**
Run the agent's browser/desktop **inside a disposable VM or container**. No host access. Fresh image per task. The big providers ship reference images:
- Anthropic: `anthropic-quickstarts/computer-use-demo` (Dockerised VNC desktop).
- OpenAI Operator: hosted, fully sandboxed (you never see the VM).
- Browserbase / Anchorbrowser: managed Playwright sandboxes.

**Layer 2 — Allowlist + denylist.**
- Restrict **URLs** the browser can navigate to (`navigate` tool checks against allowlist).
- Restrict **domains** the agent can authenticate against.
- Block known dangerous patterns (`pay`, `transfer`, `delete account`, ssh, file://).

**Layer 3 — Human-in-the-loop confirmation.**
- Any action that **moves money**, **deletes data**, **sends a message**, or **changes auth** must require explicit human approval.
- Surface a diff before "submit." Most production deployments today are **agent prepares; human clicks Submit**.

### Specific rules to bake in
- ❌ Never let a computer-use agent touch your real bank, payment, or trading account.
- ❌ Never run a computer-use agent against the host machine you use day-to-day. Always a VM.
- ❌ Never expose the agent's port (VNC, CDP) to the public internet.
- ❌ Never pass long-lived credentials *into* the agent's context window — use scoped, short-lived session tokens.
- ✅ Always log every screenshot + action with a hash + timestamp. You'll want the audit trail.
- ✅ Always cap **max steps**, **max time**, **max URL-changes** per task.
- ✅ Always prompt the model with "Stop and ask for confirmation before any irreversible action."

> 🚨 **Prompt injection in the wild.** A page can contain *invisible* text instructing the agent to email your contacts a phishing link. The VLM sees pixels; the pixels can contain natural-language instructions. **Treat every web page as a hostile prompt.** Run a separate guard model that flags actions against an instruction whitelist before executing.

## 9 · Failure modes you'll see in production

| Failure | Why | Mitigation |
|---|---|---|
| **Coordinate drift** | model clicks 8 px off; misses the button | use accessibility-tree fallback; verify with post-click screenshot |
| **Infinite loop** | model "thinks" the page hasn't changed; clicks the same button forever | hash the screenshot; abort on N identical hashes |
| **Login expiry** | session times out mid-task | persistent storage state per task; explicit `relogin` action |
| **CAPTCHA wall** | bot detection trips; agent stuck | escalate to human; never train the agent to solve CAPTCHAs |
| **Pop-up modal hijack** | cookie banner / chat widget grabs focus | wait_for + dismiss heuristic; or add the modal to the allowed-actions tutorial |
| **Element ambiguity** | "click 'Continue'" — three buttons on the page named Continue | set-of-mark numbering + textual reasoning; pick by surrounding context |
| **Latency tax** | each screenshot+inference takes 2-5 s | parallelise sub-tasks where independent; cache UI screenshots; use smaller VLM for grounding |
| **Stale screenshots** | model acts on a stale frame after a route change | always screenshot AFTER waiting for `networkidle` |
| **Prompt injection from the page** | hostile page convinces agent to exfiltrate or do harm | guard model + denylist; flag any action whose target wasn't in the original task |
| **Cost runaway** | many turns × big VLM × screenshots = $$$ | max_steps cap + cost budget alert |

The honest framing: **computer-use today is fragile.** Build it with the assumption that **3-5 % of runs will need human intervention**, and design the system so that's painless.

## 10 · Decision tree — pick a stack

```
Do you need OS-level control (desktop apps, multi-window)?
   │
   ├─ Yes
   │   ├─ hosted, willing to send screenshots to a vendor? → Claude / OpenAI / Gemini computer-use
   │   └─ self-hosted only?                                 → Anthropic ref VM + local Qwen2.5-VL + UI-TARS
   │
   └─ No (browser is enough)
       │
       ├─ rapid prototype, OK with hosted brain? → browser-use + GPT-4o / Sonnet 3.5
       ├─ production, need fine control?          → Playwright + your own loop (§6)
       ├─ commercial managed?                     → Skyvern Cloud, Browserbase / Anchorbrowser
       └─ very narrow site, deterministic?        → don't use an agent — write a Playwright script
```

### What "good" looks like in production

- Tasks **scoped narrowly** ("update CRM with this lead") rather than open-ended.
- Run inside a **fresh sandbox per task**.
- **Allowlist + guard model** + human-confirm for irreversible actions.
- **Run-replay** logging: every screenshot + action stored; failures replayed for debugging.
- **SLAs framed as completion rate**, not absolute success — "92 % of tasks done; 8 % escalated."

### When to AVOID computer-use
- **An API exists** — use it. APIs are cheaper, faster, more reliable.
- **High-volume** workloads — 5 s per action doesn't survive contact with thousands of users.
- **Anything involving money / privileged data / production systems**, unless the human-in-the-loop is robust.
- **Customer-facing real-time** flows — agent latency + flakiness will hurt UX.

## ✅ Recap
- Computer-use = **VLM watches a screen, emits click/type/scroll**. Fills the API gap.
- Hosted: **Claude · OpenAI Operator · Gemini**. OSS: **browser-use · Skyvern · Stagehand**.
- The action space is small and standardised — `screenshot · click · type · scroll · key · navigate · done`.
- **Pure UI grounding (ScreenSpot) is solved**; **end-to-end tasks (OSWorld, WebArena) are not**.
- Build the **three-layer safety stack**: VM isolation + allowlist + human-in-the-loop confirmation.
- Treat every page as a hostile prompt; run a **guard model** alongside the agent.
- Pick the simplest tool that works: **Playwright script > browser-use > full computer-use**. Reach right when you must.

Next: **M73 — Spark + Delta Lake**.
